### Day 3
Compare **LLM-A** (`llama-3.1-70b-versatile`) and **LLM-B** (`llama-3.1-8b-instant`) using:

1. **Faithfulness**: Is the answer derived solely from the provided context? (Prevents Hallucinations)
2. **Answer Relevance**: Does the answer directly address the user's question?
3. **Context Precision**: Is the retrieved context relevant and used accurately?
4. **Answer Correctness**: How close is the model's answer to the provided Ground Truth?

In [15]:
!pip install --upgrade pypdf groq pandas

In [16]:
import pypdf
import pandas as pd

def extract_text_from_pdfs(file_paths):
    combined_text = ""
    for path in file_paths:
        try:
            reader = pypdf.PdfReader(path)
            # Extracting first 50 pages from each to stay within context limits while ensuring breadth
            for i in range(min(len(reader.pages), 50)):
                combined_text += reader.pages[i].extract_text() + "\n"
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return combined_text

# Updated file list with the requested PDFs
pdf_files = [
    '/content/Seerat e Mustafa_new.pdf',
    '/content/TheSealedNectar-Alhamdulillah-library.blogspot.in.pdf'
]
raw_context = extract_text_from_pdfs(pdf_files)
print(f"Extracted {len(raw_context)} characters from the selected PDFs.")

Extracted 142611 characters from the selected PDFs.


In [19]:
import json
import os
import pypdf
from groq import Groq
from google.colab import userdata

# Initialize the Groq client using the secret key stored in Colab
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def extract_text_from_pdfs(file_paths):
    combined_text = ""
    for path in file_paths:
        try:
            reader = pypdf.PdfReader(path)
            for i in range(min(len(reader.pages), 50)):
                combined_text += reader.pages[i].extract_text() + "\n"
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return combined_text

def generate_test_set(context, num_questions=25):
    # Check if we already have a test_set in memory to avoid Rate Limits
    if 'test_set' in globals() and len(globals()['test_set']) > 0:
        print(f"Using existing test set with {len(globals()['test_set'])} questions.")
        return globals()['test_set']

    if not context:
        raise ValueError("Context is empty.")

    prompt = f"""Generate exactly {num_questions} unique factual questions and their correct ground truth answers based ONLY on the provided text.
    Return the result as a JSON object with a key 'test_set' containing a list of objects with 'question' and 'ground_truth'.

    Context: {context[:8000]}"""

    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )
        data = json.loads(completion.choices[0].message.content)
        return data.get('test_set', [])
    except Exception as e:
        print(f"Error during question generation: {e}")
        return []

# Ensure raw_context exists
if 'raw_context' not in locals() or not raw_context:
    pdf_files = ['/content/Seerat e Mustafa_new.pdf', '/content/TheSealedNectar-Alhamdulillah-library.blogspot.in.pdf']
    raw_context = extract_text_from_pdfs(pdf_files)

# Attempt to generate or recover test set
test_set = generate_test_set(raw_context, num_questions=25)

if not test_set and 'eval_data' in locals():
    print("Rate limit hit, but recovering questions from existing evaluation data...")
    test_set = [{'question': r['question'], 'ground_truth': r['ground_truth']} for r in eval_data]

print(f"Current test_set size: {len(test_set)}")

Error during question generation: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvcm6gjxe8m8zms2jb096mvw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99426, Requested 1324. Please try again in 10m48s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Rate limit hit, but recovering questions from existing evaluation data...
Current test_set size: 26


In [20]:
def evaluate_models(test_set, context):
    results = []
    models = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

    for item in test_set:
        q = item['question']
        gt = item['ground_truth']
        row = {"question": q, "ground_truth": gt}

        for model_name in models:
            try:
                # Generate Answer
                resp = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": f"Context: {context[:4000]}\nQuestion: {q}"}]
                ).choices[0].message.content

                # Judge with Llama 3.3 70B
                eval_prompt = f"""Score the following response (0.0 to 1.0) based on these 4 metrics:\n                1. Faithfulness\n                2. Answer Relevance\n                3. Context Precision\n                4. Answer Correctness\n\n                Question: {q}\n                Ground Truth: {gt}\n                Context: {context[:2000]}\n                Response: {resp}\n                \n                Return JSON: {{'faithfulness': float, 'relevance': float, 'precision': float, 'correctness': float}}"""

                eval_res = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": eval_prompt}],
                    response_format={"type": "json_object"}
                ).choices[0].message.content

                row[f"{model_name}_answer"] = resp
                row[f"{model_name}_metrics"] = json.loads(eval_res)
            except Exception as e:
                print(f"Error with {model_name}: {e}")
                row[f"{model_name}_metrics"] = {"faithfulness": 0, "relevance": 0, "precision": 0, "correctness": 0}

        results.append(row)
    return results

if 'test_set' in locals() and test_set:
    eval_data = evaluate_models(test_set, raw_context)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    display(pd.DataFrame(eval_data))
else:
    print("test_set not found. Please run the generation cell first.")

Error with llama-3.3-70b-versatile: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvcm6gjxe8m8zms2jb096mvw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99395, Requested 787. Please try again in 2m37.248s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error with llama-3.1-8b-instant: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvcm6gjxe8m8zms2jb096mvw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99395, Requested 683. Please try again in 1m7.391999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error with llama-3.3-70b-versatile: Error code: 429 - {'error': {'message': 'Rate limit reached for model `lla

,question,ground_truth,llama-3.3-70b-versatile_metrics,llama-3.1-8b-instant_metrics,llama-3.1-8b-instant_answer
0,What is the title of the biography of the Chosen Messenger of Allah?,Seeratul Mustafa,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
1,Who is the author of Seeratul Mustafa?,Hadhrat Moulana Muhammad Idrees Kaandhlawi,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
2,Who translated Seeratul Mustafa?,Mufti Muhammad Kadwa and Moulana Mahommed Mahommedy,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
3,In which year was the first edition of Seeratul Mustafa published?,2014,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
4,What is the name of the publisher of Seeratul Mustafa?,Jamiatul Ulama (KZN) Ta’limi Board,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
5,What is the address of the publisher of Seeratul Mustafa?,"4 Third Avenue, P.O.Box 26024, Isipingo Beach, 4115, South Africa","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
6,What is the phone number of the publisher of Seeratul Mustafa?,(+27) 31 912 2172 - Ext: 209,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
7,What is the email address of the publisher of Seeratul Mustafa?,info@talimiboardkzn.org,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 1.0, 'relevance': 1.0, 'precision': 1.0, 'correctness': 1.0}",info@talimiboardkzn.org
8,What is the website of the publisher of Seeratul Mustafa?,www.talimiboardkzn.org,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN
9,Who wrote the foreword for Seeratul Mustafa?,Hadhrat Moulana Abdullah Amejee Saahib and Hadhrat Maulana Ashraf ‘Ali Thaanwi,"{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}","{'faithfulness': 0, 'relevance': 0, 'precision': 0, 'correctness': 0}",NaN


In [ ]:
comparison = {}
# Update to match the models actually evaluated
active_models = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

for model in active_models:
    metrics_list = [r.get(f"{model}_metrics", {}) for r in eval_data]
    if metrics_list:
        avg_metrics = {
            "Faithfulness": sum(m.get('faithfulness', 0) for m in metrics_list) / len(metrics_list),
            "Relevance": sum(m.get('relevance', 0) for m in metrics_list) / len(metrics_list),
            "Precision": sum(m.get('precision', 0) for m in metrics_list) / len(metrics_list),
            "Correctness": sum(m.get('correctness', 0) for m in metrics_list) / len(metrics_list)
        }
        comparison[model] = avg_metrics

comparison_df = pd.DataFrame(comparison).T
print("--- Aggregate Model Performance Comparison ---")
display(comparison_df)